In [9]:
import pandas as pd
import glob, os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW = "../data/raw"
tables = {}
for f in sorted(glob.glob(os.path.join(RAW, "*.csv"))):
    name = os.path.splitext(os.path.basename(f))[0]
    tables[name] = pd.read_csv(f)

for name, df in tables.items():
    print(f"\n===== {name}  {df.shape} =====")
    print(list(df.columns))


===== carriers.csv  (4, 2) =====
['CarrierID', 'CarrierName']

===== fulfillment_status.csv  (5000, 4) =====
['FulfillmentID', 'SalesOrderID', 'ShipmentID', 'Status']

===== purchase_order_lines.csv  (3141, 5) =====
['POLineID', 'PO_ID', 'MaterialID', 'Quantity', 'UnitCost']

===== purchase_orders.csv  (1250, 6) =====
['PO_ID', 'SupplierID', 'OrderDate', 'ExpectedDeliveryDate', 'Status', 'TotalCost']

===== raw_materials.csv  (4, 4) =====
['id', 'name', 'category', 'unitCost']

===== sales_orders.csv  (5000, 6) =====
['SalesOrderID', 'OrderDate', 'ProductID', 'ProductName', 'Quantity', 'CustomerID']

===== shipment_tracking.csv  (5000, 6) =====
['ShipmentID', 'CarrierID', 'OriginWarehouseID', 'DestinationAddress', 'ShipDate', 'Status']

===== stock_movements.csv  (712, 6) =====
['MovementID', 'MaterialID', 'WarehouseID', 'MovementType', 'Quantity', 'Date']

===== supplier_performance.csv  (4, 5) =====
['SupplierID', 'SupplierName', 'OnTimeDeliveryRate', 'QualityScore', 'AverageLeadTim

In [10]:
clean = {}
log = []

for name, df in tables.items():
    d = df.copy()
    rows_before = len(d)

    # 1. Strip spaces from all text columns
    for col in d.select_dtypes(include="object").columns:
        d[col] = d[col].str.strip()

    # 2. Convert date columns to real dates
    for col in d.columns:
        if "date" in col.lower():
            before = d[col].isna().sum()
            d[col] = pd.to_datetime(d[col], errors="coerce")
            bad = d[col].isna().sum() - before
            if bad:
                print(f"WARNING {name}.{col}: {bad} values could not be read as dates")

    # 3. Remove exact duplicate rows
    d = d.drop_duplicates().reset_index(drop=True)

    clean[name] = d
    log.append({
        "table": name,
        "rows_before": rows_before,
        "duplicates_removed": rows_before - len(d),
        "rows_after": len(d),
    })

pd.DataFrame(log)

,table,rows_before,duplicates_removed,rows_after
0,carriers.csv,4,0,4
1,fulfillment_status.csv,5000,190,4810
2,purchase_order_lines.csv,3141,131,3010
3,purchase_orders.csv,1250,44,1206
4,raw_materials.csv,4,0,4
5,sales_orders.csv,5000,184,4816
6,shipment_tracking.csv,5000,189,4811
7,stock_movements.csv,712,25,687
8,supplier_performance.csv,4,0,4
9,suppliers.csv,4,1,3


In [11]:
missing = []
for name, d in clean.items():
    for col in d.columns:
        n = int(d[col].isna().sum())
        if n:
            missing.append({
                "table": name,
                "column": col,
                "missing": n,
                "percent": round(n / len(d) * 100, 1),
            })

print(pd.DataFrame(missing).to_string(index=False))

print("\n--- suppliers ---")
print(clean["suppliers.csv"])
print("\n--- supplier_performance ---")
print(clean["supplier_performance.csv"])

                   table               column  missing  percent
  fulfillment_status.csv        FulfillmentID      205      4.3
  fulfillment_status.csv         SalesOrderID      186      3.9
  fulfillment_status.csv           ShipmentID      186      3.9
  fulfillment_status.csv               Status      200      4.2
purchase_order_lines.csv             POLineID      121      4.0
purchase_order_lines.csv                PO_ID      119      4.0
purchase_order_lines.csv           MaterialID      110      3.7
purchase_order_lines.csv             Quantity       80      2.7
purchase_order_lines.csv             UnitCost       97      3.2
     purchase_orders.csv                PO_ID       48      4.0
     purchase_orders.csv           SupplierID       51      4.2
     purchase_orders.csv            OrderDate       44      3.6
     purchase_orders.csv ExpectedDeliveryDate       46      3.8
     purchase_orders.csv               Status       57      4.7
     purchase_orders.csv            Tota

In [12]:
# Make table names simple (no .csv)
clean = {k.replace(".csv", ""): v for k, v in clean.items()}
fixes = []

def drop_missing(name, cols):
    before = len(clean[name])
    clean[name] = clean[name].dropna(subset=cols).reset_index(drop=True)
    fixes.append({"table": name,
                  "action": "dropped rows missing " + ", ".join(cols),
                  "rows_affected": before - len(clean[name])})

def fill_unknown(name, cols):
    for col in cols:
        n = int(clean[name][col].isna().sum())
        clean[name][col] = clean[name][col].fillna("Unknown")
        fixes.append({"table": name,
                      "action": f"filled {col} with 'Unknown'",
                      "rows_affected": n})

# 1. Restore the missing supplier from supplier_performance
sp, sup = clean["supplier_performance"], clean["suppliers"]
for sid in set(sp["SupplierID"]) - set(sup["id"]):
    nm = sp.loc[sp["SupplierID"] == sid, "SupplierName"].iloc[0]
    sup.loc[len(sup)] = [sid, nm, "Unknown"]
    fixes.append({"table": "suppliers", "action": f"restored {sid} ({nm})", "rows_affected": 1})

# 2. Recover ProductID <-> ProductName in sales_orders
so = clean["sales_orders"]
pairs = so.dropna(subset=["ProductID", "ProductName"]).drop_duplicates(["ProductID", "ProductName"])
print("Distinct ProductID/ProductName pairs:", len(pairs),
      "| distinct IDs:", pairs["ProductID"].nunique(),
      "| distinct names:", pairs["ProductName"].nunique())
id_to_name = dict(zip(pairs["ProductID"], pairs["ProductName"]))
name_to_id = dict(zip(pairs["ProductName"], pairs["ProductID"]))
so["ProductName"] = so["ProductName"].fillna(so["ProductID"].map(id_to_name))
so["ProductID"] = so["ProductID"].fillna(so["ProductName"].map(name_to_id))

# 3. Drop rows missing join keys
drop_missing("fulfillment_status", ["SalesOrderID", "ShipmentID"])
drop_missing("purchase_order_lines", ["PO_ID", "MaterialID"])
drop_missing("purchase_orders", ["PO_ID", "SupplierID"])
drop_missing("sales_orders", ["SalesOrderID", "ProductID"])
drop_missing("shipment_tracking", ["ShipmentID"])
drop_missing("stock_movements", ["MaterialID", "WarehouseID", "MovementType"])

# 4. Fill text columns with 'Unknown'
fill_unknown("fulfillment_status", ["Status"])
fill_unknown("purchase_orders", ["Status"])
fill_unknown("sales_orders", ["CustomerID"])
fill_unknown("shipment_tracking", ["CarrierID", "OriginWarehouseID", "DestinationAddress", "Status"])

print(pd.DataFrame(fixes).to_string(index=False))
print()
for name, d in clean.items():
    print(f"{name}: {len(d)} rows")

Distinct ProductID/ProductName pairs: 4 | distinct IDs: 4 | distinct names: 4
               table                                                     action  rows_affected
           suppliers                         restored SUP02 (Global Sweeteners)              1
  fulfillment_status              dropped rows missing SalesOrderID, ShipmentID            364
purchase_order_lines                     dropped rows missing PO_ID, MaterialID            225
     purchase_orders                     dropped rows missing PO_ID, SupplierID             94
        sales_orders               dropped rows missing SalesOrderID, ProductID            158
   shipment_tracking                            dropped rows missing ShipmentID            203
     stock_movements dropped rows missing MaterialID, WarehouseID, MovementType             83
  fulfillment_status                               filled Status with 'Unknown'            173
     purchase_orders                               filled Status wi

In [13]:
fixes = []
def note(table, action, n):
    fixes.append({"table": table, "action": action, "rows_affected": int(n)})

rm  = clean["raw_materials"]
pol = clean["purchase_order_lines"]

# 1. Median unit cost per material (from PO lines)
med = pol.groupby("MaterialID")["UnitCost"].median()

mask = rm["unitCost"].isna()
rm.loc[mask, "unitCost"] = rm.loc[mask, "id"].map(med)
note("raw_materials", "filled unitCost with median from PO lines", mask.sum())

n = pol["UnitCost"].isna().sum()
pol["UnitCost"] = pol["UnitCost"].fillna(pol["MaterialID"].map(med))
note("purchase_order_lines", "filled UnitCost with material median", n)

# 2. Drop rows missing Quantity or the main date
def drop_na(name, cols):
    before = len(clean[name])
    clean[name] = clean[name].dropna(subset=cols).reset_index(drop=True)
    note(name, "dropped rows missing " + ", ".join(cols), before - len(clean[name]))

drop_na("purchase_order_lines", ["Quantity"])
drop_na("sales_orders", ["Quantity", "OrderDate"])
drop_na("stock_movements", ["Quantity", "Date"])
drop_na("purchase_orders", ["OrderDate"])

# 3. Infer the missing WarehouseID in warehouse_inventory
wi = clean["warehouse_inventory"]
for idx, row in wi[wi["WarehouseID"].isna()].iterrows():
    used = set(wi.loc[wi["MaterialID"] == row["MaterialID"], "WarehouseID"].dropna())
    options = set(clean["warehouses"]["id"]) - used
    if len(options) == 1:
        wi.loc[idx, "WarehouseID"] = options.pop()
        note("warehouse_inventory", "inferred missing WarehouseID", 1)

print(pd.DataFrame(fixes).to_string(index=False))

# 4. Sanity checks: ranges and odd values
print("\n--- numeric ranges ---")
for name, col in [("purchase_order_lines", "Quantity"), ("purchase_order_lines", "UnitCost"),
                  ("purchase_orders", "TotalCost"), ("sales_orders", "Quantity"),
                  ("stock_movements", "Quantity")]:
    s = clean[name][col]
    print(f"{name}.{col}: min={s.min()}, median={s.median()}, max={s.max()}, zero_or_negative={(s <= 0).sum()}")

print("\n--- date ranges ---")
for name, col in [("sales_orders", "OrderDate"), ("purchase_orders", "OrderDate"),
                  ("shipment_tracking", "ShipDate"), ("stock_movements", "Date")]:
    s = clean[name][col]
    print(f"{name}.{col}: {s.min().date()} to {s.max().date()}")

print("\nMovementType values:", clean["stock_movements"]["MovementType"].unique())

               table                                    action  rows_affected
       raw_materials filled unitCost with median from PO lines              1
purchase_order_lines      filled UnitCost with material median             87
purchase_order_lines             dropped rows missing Quantity             71
        sales_orders  dropped rows missing Quantity, OrderDate            299
     stock_movements       dropped rows missing Quantity, Date             51
     purchase_orders            dropped rows missing OrderDate             43
 warehouse_inventory              inferred missing WarehouseID              1

--- numeric ranges ---
purchase_order_lines.Quantity: min=500.0, median=1254.5, max=2000.0, zero_or_negative=0
purchase_order_lines.UnitCost: min=0.5, median=5.1, max=20.5, zero_or_negative=0
purchase_orders.TotalCost: min=253.0, median=23782.1, max=107437.25, zero_or_negative=0
sales_orders.Quantity: min=10.0, median=105.0, max=200.0, zero_or_negative=0
stock_movements.Qu

In [14]:
today = pd.Timestamp("2026-11-11")

# 1. Future dates
print("--- rows dated after 11 Nov 2026 ---")
for name, col in [("sales_orders", "OrderDate"), ("purchase_orders", "OrderDate"),
                  ("shipment_tracking", "ShipDate"), ("stock_movements", "Date")]:
    n = (clean[name][col] > today).sum()
    print(f"{name}.{col}: {n} rows")

# 2. TotalCost vs sum of PO lines
pol = clean["purchase_order_lines"]
po = clean["purchase_orders"]
line_total = (pol["Quantity"] * pol["UnitCost"]).groupby(pol["PO_ID"]).sum()

m = po.dropna(subset=["TotalCost"]).copy()
m["calc"] = m["PO_ID"].map(line_total)
has_lines = m["calc"].notna()
match = (m["calc"] - m["TotalCost"]).abs() < 1

print("\n--- TotalCost check ---")
print("POs with a TotalCost:", len(m))
print("  of which have order lines:", int(has_lines.sum()))
print("  of which TotalCost matches the lines:", int((has_lines & match).sum()))
print("POs with missing TotalCost:", int(po["TotalCost"].isna().sum()))
print("POs with no lines at all:", int((~po["PO_ID"].isin(pol["PO_ID"])).sum()))

# 3. Orphan keys
print("\n--- orphan keys (values with no matching record) ---")
checks = [
    ("purchase_orders", "SupplierID", "suppliers", "id"),
    ("purchase_order_lines", "PO_ID", "purchase_orders", "PO_ID"),
    ("purchase_order_lines", "MaterialID", "raw_materials", "id"),
    ("stock_movements", "MaterialID", "raw_materials", "id"),
    ("stock_movements", "WarehouseID", "warehouses", "id"),
    ("fulfillment_status", "SalesOrderID", "sales_orders", "SalesOrderID"),
    ("fulfillment_status", "ShipmentID", "shipment_tracking", "ShipmentID"),
    ("shipment_tracking", "CarrierID", "carriers", "CarrierID"),
    ("shipment_tracking", "OriginWarehouseID", "warehouses", "id"),
]
for t, c, rt, rc in checks:
    vals = clean[t][c]
    vals = vals[vals != "Unknown"]
    orphans = (~vals.isin(clean[rt][rc])).sum()
    print(f"{t}.{c} -> {rt}.{rc}: {orphans} orphans out of {len(vals)}")

--- rows dated after 11 Nov 2026 ---
sales_orders.OrderDate: 0 rows
purchase_orders.OrderDate: 0 rows
shipment_tracking.ShipDate: 2 rows
stock_movements.Date: 27 rows

--- TotalCost check ---
POs with a TotalCost: 1022
  of which have order lines: 979
  of which TotalCost matches the lines: 0
POs with missing TotalCost: 47
POs with no lines at all: 45

--- orphan keys (values with no matching record) ---
purchase_orders.SupplierID -> suppliers.id: 0 orphans out of 1069
purchase_order_lines.PO_ID -> purchase_orders.PO_ID: 421 orphans out of 2714
purchase_order_lines.MaterialID -> raw_materials.id: 0 orphans out of 2714
stock_movements.MaterialID -> raw_materials.id: 0 orphans out of 553
stock_movements.WarehouseID -> warehouses.id: 0 orphans out of 553
fulfillment_status.SalesOrderID -> sales_orders.SalesOrderID: 590 orphans out of 4446
fulfillment_status.ShipmentID -> shipment_tracking.ShipmentID: 380 orphans out of 4446
shipment_tracking.CarrierID -> carriers.CarrierID: 0 orphans out 

In [15]:
today = pd.Timestamp("2026-11-11")
fixes = []
def note(table, action, n):
    fixes.append({"table": table, "action": action, "rows_affected": int(n)})

# 1. Drop rows with future dates
for name, col in [("shipment_tracking", "ShipDate"), ("stock_movements", "Date")]:
    before = len(clean[name])
    clean[name] = clean[name][~(clean[name][col] > today)].reset_index(drop=True)
    note(name, f"dropped rows with {col} after 11 Nov 2026", before - len(clean[name]))

# 2. Cost columns
pol = clean["purchase_order_lines"]
pol["LineTotal"] = (pol["Quantity"] * pol["UnitCost"]).round(2)
note("purchase_order_lines", "added LineTotal = Quantity x UnitCost", len(pol))

clean["purchase_orders"] = clean["purchase_orders"].rename(columns={"TotalCost": "ReportedTotalCost"})
note("purchase_orders", "renamed TotalCost to ReportedTotalCost (did not match lines)", 0)

# 3. Consistent key names in lookup tables
clean["suppliers"] = clean["suppliers"].rename(columns={"id": "SupplierID", "name": "SupplierName"})
clean["warehouses"] = clean["warehouses"].rename(columns={"id": "WarehouseID", "name": "WarehouseName"})
clean["raw_materials"] = clean["raw_materials"].rename(
    columns={"id": "MaterialID", "name": "MaterialName", "unitCost": "UnitCost"})
note("lookup tables", "renamed id/name columns to SupplierID, WarehouseID, MaterialID, etc.", 0)

# 4. Save cleaned files
os.makedirs("../data/cleaned", exist_ok=True)
summary = []
for name, d in clean.items():
    d.to_csv(f"../data/cleaned/{name}_clean.csv", index=False)
    summary.append({"file": f"{name}_clean.csv", "rows": len(d), "columns": d.shape[1]})

print(pd.DataFrame(fixes).to_string(index=False))
print()
print(pd.DataFrame(summary).to_string(index=False))

               table                                                               action  rows_affected
   shipment_tracking                         dropped rows with ShipDate after 11 Nov 2026              2
     stock_movements                             dropped rows with Date after 11 Nov 2026             27
purchase_order_lines                                added LineTotal = Quantity x UnitCost           2714
     purchase_orders         renamed TotalCost to ReportedTotalCost (did not match lines)              0
       lookup tables renamed id/name columns to SupplierID, WarehouseID, MaterialID, etc.              0

                          file  rows  columns
            carriers_clean.csv     4        2
  fulfillment_status_clean.csv  4446        4
purchase_order_lines_clean.csv  2714        6
     purchase_orders_clean.csv  1069        6
       raw_materials_clean.csv     4        4
        sales_orders_clean.csv  4359        6
   shipment_tracking_clean.csv  4606        6
 